# Verification Notebook 5
# Multi-Model Switching with Contact

In [ ]:
from pathlib import Path
import sys
_repo = Path.cwd()
while _repo != _repo.parent and not (_repo / "pyproject.toml").exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

In [ ]:
import numpy as np

from pathlib import Path
from matplotlib.ticker import ScalarFormatter
from plot_setup import METRIC_COLORS, REFERENCE_COLOR, STEP_SIZE_COLORS
from plot_setup import set_professional_style, FULL_WIDTH
plt = set_professional_style(latex=True)

from demos.ControlledPendulum.src.master_pendulum import MasterPendulum
import demos.ControlledPendulum.src.master_pendulum.components.fem.pendulum_config as config
from syssimx import FMUComponent, System, SystemGraphVisualizer, Connection, EventConnection

## Discover FMUs

In [ ]:
PLATFORM = sys.platform
demo_dir_path = _repo / "demos" / "ControlledPendulum"
package_path = Path(demo_dir_path / "src/modelica/ControlledPendulum")
fmu_output_dir = Path(demo_dir_path / f"artifacts/fmus/{PLATFORM}")

fmu_paths = {}
for subdir in fmu_output_dir.iterdir():
    if subdir.is_dir():
        fmu_paths[subdir.name] = {}
        for fmu_file in subdir.glob("*.fmu"):
            fmu_paths[subdir.name][fmu_file.stem] = fmu_file
    else:
        fmu_paths[subdir.stem] = subdir


## Modelica Reference Simulation

In [ ]:
from OMPython import ModelicaSystem

reference = ModelicaSystem(fileName=str(package_path / "package.mo"),
                           modelName="ControlledPendulum.Examples.Contact.RigidContact")
reference.setParameters({'useReset':'true'})
reference.buildModel()
reference.simulate()
ref_sol_names = ('time', 'theta', 'theta_ref', 'theta_meas', 'u_control', 'pid.I_out', 'pendulum.contact')
ref_sol_reset = {name: reference.getSolutions(name).flatten() for name in ref_sol_names}

## Instantiate FMU Components

In [ ]:
def print_parameteres(component: FMUComponent):
    print(f"Parameters of {component.name}:")
    for name, param in component.parameters.items():
        if '.' not in name and param is not None:
            print(f"  - {name}: {param}")

## Components

In [ ]:
setpoint = FMUComponent(name="Setpoint",
                         fmu_path=fmu_paths['Trajectories']["SetPoint"],
                         group="Reference")

class PIDController(FMUComponent):
    def __init__(self, name):
        fmu_path = fmu_paths['Controllers']['PIDControllerReset_euler']
        super().__init__(name=name, fmu_path=fmu_path, group="Controller")
        self.use_reset = False  # Flag to control whether to apply reset on event
    
    def _handle_events_internal(self, event_names, t):
        if "wall_hit" not in event_names:
            return
        self.set_inputs({"resetI": True})
        self.do_step(t, 0)  # Perform a step to apply the resetI input
        self.set_inputs({"resetI": False})

pid = PIDController(name="PID")

drive = FMUComponent(name="Drive",
                     fmu_path=fmu_paths['Actuators']["DriveDynamic"],
                     group="Actuator")

sim_params = config.SimulationParameters()
sim_params.tau = 0.001
sim_params.t_end = 0.4
sim_params.with_contact = True
sim_params.use_gravity = True

anim_params = config.AnimationParameters()
anim_params.animate = False

fem_parameters = {
    'sim_params': sim_params,
    'anim_params': anim_params,
}

pendulum = MasterPendulum(name="MasterPendulum", initial_mode="FMU")
pendulum.record_switch_state = True
pendulum.set_parameters(**{"FEM": fem_parameters})
pendulum.initialize(t0=0.0)
pendulum._allow_mode_switching = True

def wall_contact_indicator(comp: MasterPendulum) -> float:
    theta = comp.get_outputs()["theta"]
    theta_wall = 0.0
    return theta - theta_wall

pendulum.add_event_indicator("wall_hit", func=wall_contact_indicator, direction=-1)

angle_sensor = FMUComponent(name="Angle Sensor",
                             fmu_path=fmu_paths['Sensors']['AngleSensor'],
                             group="Sensors")

angle_decoder = FMUComponent(name="Angle Decoder",
                             fmu_path=fmu_paths['Sensors']['AngleDecoder'],
                             group="Signal Processing")

## Connections

In [ ]:
c1 = Connection(
    src_comp=setpoint.name,
    src_port=setpoint.output_specs["theta_ref"].name,
    dst_comp=pid.name,
    dst_port=pid.input_specs["theta_ref"].name,
)

c21 = Connection(
    src_comp=pendulum.name,
    src_port=pendulum.output_specs["theta"].name,
    dst_comp=angle_sensor.name,
    dst_port=angle_sensor.input_specs["theta"].name,
)

c22 = Connection(
    src_comp=angle_sensor.name,
    src_port=angle_sensor.output_specs["v_out"].name,
    dst_comp=angle_decoder.name,
    dst_port=angle_decoder.input_specs["v_in"].name,
)

c23 = Connection(
    src_comp=angle_decoder.name,
    src_port=angle_decoder.output_specs["theta"].name,
    dst_comp=pid.name,
    dst_port=pid.input_specs["theta_meas"].name,
)

c3 = Connection(
    src_comp=pid.name,
    src_port=pid.output_specs["u"].name,
    dst_comp=drive.name,
    dst_port=drive.input_specs["u_control"].name,
)

c4 = Connection(
    src_comp=drive.name,
    src_port=drive.output_specs["torque"].name,
    dst_comp=pendulum.name,
    dst_port=pendulum.input_specs["tau"].name,
)

c5 = Connection(
    src_comp=pendulum.name,
    src_port=pendulum.output_specs["omega"].name,
    dst_comp=drive.name,
    dst_port=drive.input_specs["omega"].name,
)

event_connection_1 = EventConnection(
    src_comp=pendulum.name,
    src_port="wall_hit",
    dst_comp=pendulum.name,
    dst_port=pendulum.input_specs["omega_invert"].name,
)

event_connection_2 = EventConnection(
    src_comp=pendulum.name,
    src_port="wall_hit",
    dst_comp=pid.name,
    dst_port=pid.input_specs["resetI"].name,
)

connections = [c1, c21, c22, c23, c3, c4, c5]
components  = [setpoint, pid, drive, pendulum, angle_decoder, angle_sensor]

system = System(name="Pendulum System with Master Pendulum")

for comp in components:
    system.add_component(comp)

for conn in connections:
    system.add_connection(conn)

system.add_event_connection(event_connection_1)
system.add_event_connection(event_connection_2)

system.initialize(t0=0.0)

In [ ]:
system.algorithm.record_internal_steps = False
system.algorithm.event_dedup_tol = 5e-4
system.algorithm.tol_time = 1e-5
system.algorithm.tol_value = 5e-5

## Run Simulation

In [ ]:
t = 0.0
dt = sim_params.tau
t_end = sim_params.t_end

system.run(t, t_end, dt)

In [ ]:
# system_res_file = _repo / "demos/ControlledPendulum/artifacts/results/master_pendulum_system.csv"
# system_res_file.parent.mkdir(parents=True, exist_ok=True)
# system.history.save_csv(filepath=system_res_file)

## Collect Results

In [ ]:
history = system.get_history()
fem_hist = pendulum.fem.get_history()
fmu_hist = pendulum.fmu.get_history()
opensim_hist = pendulum.opensim.get_history()

ref_history = history["Setpoint"]
setpoint_time, data = ref_history
theta_ref = data["theta_ref"]

t_fem = fem_hist['theta']['time']
theta_fem = fem_hist['theta']['values']
t_fmu = fmu_hist['theta']['time']
theta_fmu = fmu_hist['theta']['values']
t_opensim = opensim_hist['theta']['time']
theta_opensim = opensim_hist['theta']['values']

t_pid = history["PID"][0]
i_out = history["PID"][1]['I_out']
u_pid = history["PID"][1]['u']

event_history = history['Events']
event_times_dense = event_history[('MasterPendulum', 'wall_hit')]
event_times_float = [dt.t for dt in event_times_dense]#[:-1]
event_indices = np.where(ref_sol_reset['pendulum.contact'] > 0)[0]
event_times_ref_indices = np.where(ref_sol_reset['time'][event_indices] <= t_end)[0]
event_times_ref = ref_sol_reset['time'][event_indices][event_times_ref_indices][::2]

diff_event_times = np.array(event_times_float) - np.array(event_times_ref)

In [ ]:
import numpy as np
import pandas as pd

def trajectory_error_metrics(t, y, t_ref, y_ref, *, t_min=None, t_max=None):
    """Compute sampled trajectory errors against an interpolated reference."""
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    t_ref = np.asarray(t_ref, dtype=float)
    y_ref = np.asarray(y_ref, dtype=float)

    mask = np.ones_like(t, dtype=bool)
    if t_min is not None:
        mask &= t >= t_min
    if t_max is not None:
        mask &= t <= t_max

    t_eval = t[mask]
    y_eval = y[mask]
    y_ref_eval = np.interp(t_eval, t_ref, y_ref)

    err = y_eval - y_ref_eval
    duration = t_eval[-1] - t_eval[0]

    return {
        "e_inf": np.max(np.abs(err)),
        "e_2": np.sqrt(np.trapezoid(err**2, t_eval) / duration),
        "e_mean": np.mean(np.abs(err)),
        "n_samples": len(t_eval),
    }


def event_time_metrics(t_events, t_events_ref):
    """Compare event times pairwise."""
    t_events = np.asarray(t_events, dtype=float)
    t_events_ref = np.asarray(t_events_ref, dtype=float)

    n = min(len(t_events), len(t_events_ref))
    delta = t_events[:n] - t_events_ref[:n]

    return {
        "n_syssimx": len(t_events),
        "n_reference": len(t_events_ref),
        "n_compared": n,
        "max_abs_delta": np.max(np.abs(delta)) if n else np.nan,
        "mean_abs_delta": np.mean(np.abs(delta)) if n else np.nan,
        "delta": delta,
    }


# Main switched plant output.
t_master, data_master = history["MasterPendulum"]
theta_master = np.asarray(data_master["theta"], dtype=float)

# Optional diagnostic signal.
t_pid = np.asarray(t_pid, dtype=float)
i_out = np.asarray(i_out, dtype=float)

# Full horizon errors.
theta_metrics_full = trajectory_error_metrics(
    t_master,
    theta_master,
    ref_sol_reset["time"],
    ref_sol_reset["theta"],
)

pid_metrics_full = trajectory_error_metrics(
    t_pid,
    i_out,
    ref_sol_reset["time"],
    ref_sol_reset["pid.I_out"],
)

# Contact-window errors.
# Use a slightly wider interval than the plotted contact window.
contact_window = (0.15, 0.35)

theta_metrics_contact = trajectory_error_metrics(
    t_master,
    theta_master,
    ref_sol_reset["time"],
    ref_sol_reset["theta"],
    t_min=contact_window[0],
    t_max=contact_window[1],
)

pid_metrics_contact = trajectory_error_metrics(
    t_pid,
    i_out,
    ref_sol_reset["time"],
    ref_sol_reset["pid.I_out"],
    t_min=contact_window[0],
    t_max=contact_window[1],
)

u_metrics_full = trajectory_error_metrics(
    t_pid, u_pid, ref_sol_reset["time"], ref_sol_reset["u_control"])
u_metrics_contact = trajectory_error_metrics(
    t_pid, u_pid, ref_sol_reset["time"], ref_sol_reset["u_control"],
    t_min=contact_window[0], t_max=contact_window[1])


event_metrics = event_time_metrics(event_times_float, event_times_ref)

df_metrics = pd.DataFrame(
    [
        {
            "Quantity": r"$\theta$ full horizon",
            **theta_metrics_full,
        },
        {
            "Quantity": r"$\theta$ contact window",
            **theta_metrics_contact,
        },
        {
            "Quantity": r"$I_{\mathrm{out}}$ full horizon",
            **pid_metrics_full,
        },
        {
            "Quantity": r"$I_{\mathrm{out}}$ contact window",
            **pid_metrics_contact,
        },
        {
            "Quantity": r"$u_{\mathrm{control}}$ full horizon",
            **u_metrics_full,
        },
        {
            "Quantity": r"$u_{\mathrm{control}}$ contact window",
            **u_metrics_contact,
        },
    ]
)

df_metrics


In [ ]:
from IPython.display import display

df_metrics_clean = pd.DataFrame(
    [
        {
            "Signal": "theta",
            "Interval": "full horizon",
            "E_inf": theta_metrics_full["e_inf"],
            "E_2": theta_metrics_full["e_2"],
            "E_mean": theta_metrics_full["e_mean"],
        },
        {
            "Signal": "theta",
            "Interval": "contact window",
            "E_inf": theta_metrics_contact["e_inf"],
            "E_2": theta_metrics_contact["e_2"],
            "E_mean": theta_metrics_contact["e_mean"],
        },
        {
            "Signal": "I_out",
            "Interval": "full horizon",
            "E_inf": pid_metrics_full["e_inf"],
            "E_2": pid_metrics_full["e_2"],
            "E_mean": pid_metrics_full["e_mean"],
        },
        {
            "Signal": "I_out",
            "Interval": "contact window",
            "E_inf": pid_metrics_contact["e_inf"],
            "E_2": pid_metrics_contact["e_2"],
            "E_mean": pid_metrics_contact["e_mean"],
        },
        {
            "Signal": "u_control",
            "Interval": "full horizon",
            "E_inf": u_metrics_full["e_inf"],
            "E_2": u_metrics_full["e_2"],
            "E_mean": u_metrics_full["e_mean"],
        },
        {
            "Signal": "u_control",
            "Interval": "contact window",
            "E_inf": u_metrics_contact["e_inf"],
            "E_2": u_metrics_contact["e_2"],
            "E_mean": u_metrics_contact["e_mean"],
        },
    ]
)

display(
    df_metrics_clean.style
    .format({"E_inf": "{:.3e}", "E_2": "{:.3e}", "E_mean": "{:.3e}"})
    .hide(axis="index")
)


In [ ]:
df_metrics_latex = df_metrics_clean.copy()
df_metrics_latex = df_metrics_latex.rename(
    columns={
        "E_inf": r"$E_\infty$",
        "E_2": r"$E_2$",
        "E_mean": r"$E_\mathrm{mean}$",
    }
)

print(
    df_metrics_latex.to_latex(
        index=False,
        escape=False,
        float_format="{:.3e}".format,
    )
)


In [ ]:
import pandas as pd


df_events = pd.DataFrame({
    "syssimx (s)": event_times_float,
    "Modelica (s)": event_times_ref,
    "Delta (s)": diff_event_times,
})
df_events.index.name = "Event"
df_events.style.format("{:.6f}")

In [ ]:
print(f"E_inf theta full      = {theta_metrics_full['e_inf']:.3e} rad")
print(f"E_2 theta full        = {theta_metrics_full['e_2']:.3e} rad")
print(f"E_inf theta contact   = {theta_metrics_contact['e_inf']:.3e} rad")
print(f"E_2 theta contact     = {theta_metrics_contact['e_2']:.3e} rad")
print(f"max event time error  = {event_metrics['max_abs_delta']:.3e} s")


In [ ]:
def scalar_value(value):
    if isinstance(value, dict):
        value = value.get("value", np.nan)
    if hasattr(value, "magnitude"):
        value = value.magnitude
    return float(value)


def get_state_value(state, name):
    if name in state:
        return scalar_value(state[name])
    if f"{name}_start" in state:
        return scalar_value(state[f"{name}_start"])
    return np.nan


switch_rows = []
for event in pendulum.sync_events:
    before = event["retrieved"]
    after = event["now"]

    row = {
        "t": event["time"],
        "from": event["from_mode"],
        "to": event["to_mode"],
    }

    for name in ["theta", "omega", "tau"]:
        before_value = get_state_value(before, name)
        after_value = get_state_value(after, name)
        row[f"delta_{name}"] = after_value - before_value
        row[f"abs_delta_{name}"] = abs(after_value - before_value)

    switch_rows.append(row)

df_switching = pd.DataFrame(switch_rows)

display(
    df_switching.style
    .format({
        "t": "{:.6f}",
        "delta_theta": "{:.3e}",
        "abs_delta_theta": "{:.3e}",
        "delta_omega": "{:.3e}",
        "abs_delta_omega": "{:.3e}",
        "delta_tau": "{:.3e}",
        "abs_delta_tau": "{:.3e}",
    })
    .hide(axis="index")
)


In [ ]:
def build_mode_intervals(sync_events, initial_mode, t0, tf):
    rows = []
    current_mode = initial_mode
    current_t = t0

    for event in sync_events:
        switch_t = event["time"]
        rows.append((current_t, switch_t, current_mode))
        current_t = switch_t
        current_mode = event["to_mode"]

    rows.append((current_t, tf, current_mode))
    return rows

mode_intervals = build_mode_intervals(
    pendulum.sync_events,
    initial_mode="FMU",
    t0=0.0,
    tf=t_end,
)

## Plot: Multi-Model Switching
This plot overlays all three sub-models and the setpoint.

In [ ]:
marker_size = 2
palette = {
    "fem": "#1f77b4",
    "opensim": "#ff7f0e",
    "fmu": "#2ca02c",
    "setpoint": "#d62728",
}

fem_style = {
    "color": palette["fem"],
    "linestyle": "None",
    "marker": "o",
    "markersize": marker_size,
    "linewidth": 2.5,
}
opensim_style = {
    "color": palette["opensim"],
    "linestyle": "None",
    "marker": "o",
    "markersize": marker_size,
    "linewidth": 2.5,
}
fmu_style = {
    "color": palette["fmu"],
    "linestyle": "None",
    "marker": "o",
    "markersize": marker_size,
    "linewidth": 2.5,
}

In [ ]:
fig = plt.figure(figsize=(FULL_WIDTH, 0.75 * FULL_WIDTH), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[4, 0.4, 2])
ax_full = fig.add_subplot(gs[0, 0])
ax_mode = fig.add_subplot(gs[1, 0], sharex=ax_full)
ax_contact = fig.add_subplot(gs[2, 0])

for ax in (ax_full, ax_contact):
    ax.plot(t_fem, theta_fem, label="FEM", **fem_style)
    ax.plot(t_opensim, theta_opensim, label="OpenSim", **opensim_style)
    ax.plot(t_fmu, theta_fmu, label="FMU", **fmu_style)
    ax.plot(setpoint_time, theta_ref, label="Setpoint", color=palette["setpoint"], linestyle="--", linewidth=1.5, alpha=0.55)
    ax.plot(ref_sol_reset['time'], ref_sol_reset['theta'], 'k-', label="Reference", markersize=1, linewidth=1.5, alpha=0.7)
    ax.axhline(0, color='gray', linestyle=':', linewidth=1, label="Wall Position")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(r"$\theta$ in $\mathrm{rad}$")

ax_full.set_ylim(-0.025, 0.38)
ax_full.tick_params(labelbottom=False)
ax_full.legend(loc="upper center", bbox_to_anchor=(0.635, 1), ncol=3, markerscale=2, fontsize=8)
ax_contact.set_xlabel(r"$t$ in $\mathrm{s}$")

ax_contact.set_xlim(0.15, 0.35)
ax_contact.set_ylim(-0.005, 0.07)

for t0_i, t1_i, mode in mode_intervals:
    ax_mode.broken_barh(
        [(t0_i, t1_i - t0_i)],
        (0, 1),
        facecolors=palette[mode.lower()],
        edgecolors="none",
        alpha=0.85,
    )

ax_mode.set_ylim(0, 1)
ax_mode.set_xlim(0, t_end)
ax_mode.set_yticks([])
ax_mode.grid(False)

for spine in ["left", "right", "top"]:
    ax_mode.spines[spine].set_visible(False)

for label, ax in [("(a)", ax_full), ("(b)", ax_contact)]:
    ax.text(0.02, 0.96,
        label, transform=ax.transAxes,
        ha="left", va="top",
        fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=1, pad=1.5),
    )
plt.show()

out = _repo / "notebooks" / "figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "05_a_multi_model_switching.pdf")
plt.show()


### Model Switching and Synchronicity

In [ ]:
def scalar_value(value):
    if isinstance(value, dict):
        value = value.get("value", np.nan)
    if hasattr(value, "magnitude"):
        value = value.magnitude
    return float(value)


def get_state_value(state, name):
    if name in state:
        return scalar_value(state[name])
    if f"{name}_start" in state:
        return scalar_value(state[f"{name}_start"])
    return np.nan


switch_rows = []
for event in pendulum.sync_events:
    before = event["retrieved"]
    after = event["now"]

    row = {
        "t": event["time"],
        "from": event["from_mode"],
        "to": event["to_mode"],
    }

    for name in ["theta", "omega", "tau"]:
        before_value = get_state_value(before, name)
        after_value = get_state_value(after, name)
        row[f"delta_{name}"] = after_value - before_value
        row[f"abs_delta_{name}"] = abs(after_value - before_value)

    switch_rows.append(row)

df_switching = pd.DataFrame(switch_rows)

display(
    df_switching.style
    .format({
        "t": "{:.6f}",
        "delta_theta": "{:.3e}",
        "abs_delta_theta": "{:.3e}",
        "delta_omega": "{:.3e}",
        "abs_delta_omega": "{:.3e}",
        "delta_tau": "{:.3e}",
        "abs_delta_tau": "{:.3e}",
    })
    .hide(axis="index")
)


In [ ]:
switch_summary = {
    "n_switches": len(df_switching),
    "max_abs_delta_theta": df_switching["abs_delta_theta"].max(),
    "max_abs_delta_omega": df_switching["abs_delta_omega"].max(),
    "max_abs_delta_tau": df_switching["abs_delta_tau"].max(),
}

switch_summary


## PID Controller Trajectory

In [ ]:
fig, (ax_u, ax_i) = plt.subplots(
    2, 1, sharex=True,
    figsize=(FULL_WIDTH, 0.5 * FULL_WIDTH),
    constrained_layout=True,
)

ref_style = dict(linestyle="-", color="0.4", linewidth=1.2, alpha=0.9)
cs_style  = dict(linestyle="None", marker="o", markersize=2.5, alpha=0.8,
                 color=palette["fmu"])

# (a) controller output command
ax_u.plot(t_pid, u_pid, **cs_style, label=r"\textit{SysSimX}")
ax_u.plot(ref_sol_reset["time"], ref_sol_reset["u_control"], **ref_style, label="Reference")


# (b) integrator state
ax_i.plot(t_pid, i_out, **cs_style)
ax_i.plot(ref_sol_reset["time"], ref_sol_reset["pid.I_out"], **ref_style)


# contact events on both panels, labelled once
for ax in (ax_u, ax_i):
    for k, t_event in enumerate(event_times_float):
        ax.axvline(
            t_event, color="red", linestyle=":", linewidth=1, alpha=0.5,
            label="Contact event" if (ax is ax_u and k == 0) else None,
        )

for ax in (ax_u, ax_i):
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, t_end)

ax_u.set_ylabel(r"$u_{\mathrm{control}}$")
ax_i.set_ylabel(r"$I_{\mathrm{out}}$")
ax_i.set_xlabel(r"$t$ in $\mathrm{s}$")

for label, ax in [("(a)", ax_u), ("(b)", ax_i)]:
    ax.text(
        0.012, 0.93, label, transform=ax.transAxes,
        ha="left", va="top", fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.5),
    )

fig.align_ylabels((ax_u, ax_i))

handles, labels = ax_u.get_legend_handles_labels()
fig.legend(handles, labels, ncol=3,
           fontsize=8, markerscale=2, frameon=True, loc="upper center", bbox_to_anchor=(0.52, 0.99))

out = _repo / "notebooks" / "figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "05_b_multi_model_controller.pdf")
plt.show()


---

# Unified Event-Localized Region Switching

`MultiComponent` uses one ordered region map to decide when the active model changes.

The scalar key is `abs(theta)`, with FEM, OpenSim, and FMU assigned to increasing-angle regions.
Each boundary is localized by the hybrid algorithm independently of the communication grid.

`set_switch_regions()` generates each condition as a zero-crossing. The hybrid
algorithm detects the crossing, narrows it by bisection, and applies the switch at the located
instant — the same machinery that localizes the `wall_hit` contact event.

The run above uses the default three-model region policy. This section compares two macro-step sizes
against the **OpenModelica reference** rather than against another SysSimX mechanism,
using the event-time metric already defined above.

### Why the comparison covers all three models

The region map spans FEM, OpenSim, and FMU with each physical boundary represented once.
`MultiComponent` resolves the target from boundary identity and localized crossing direction,
so adjacent transitions remain deterministic even when one macro step crosses several boundaries.
Both comparison runs therefore exercise the same three-model policy; only the macro-step size differs.

### No time gate is required

Initialization needs no time gate: θ starts at 0, which is *inside* the FEM region. Initialization-time
reconciliation therefore establishes the FEM region before stepping; subsequent changes occur only at
localized boundary crossings.

In [ ]:
SWITCH_THRESHOLD_RAD = 0.075          # FEM while |theta| <= threshold
SWITCH_BAND_RAD = 5e-3                # hysteresis half-band

# Both comparison runs use the tolerances of the main run above. That is not
# incidental — the two event families in this system want opposite things from
# `tol_time`, and this value is the one that serves both:
#
#   contact  `FEMPendulum` brackets the wall hit to its own sub-step (1e-4 s).
#            `_locate_event_time` would accept that bracket directly if
#            tol_time >= 1e-4, skipping bisection. Here it does not, so contact
#            costs ~4 extra FEM solves per event. Acceptable for one accuracy run.
#
#   switch   bisection resolves the crossing to tol_time. At 1.5e-4 that is 15 %
#            of a macro step, too coarse for useful localized placement, which defeats
#            the comparison. At 1e-5 it is 1 %.
#
# `tol_value` follows from `tol_time`: the generated region boundary is in radians and
# moves at |omega| ~ 2.5 rad/s, so bisecting to 1e-5 s leaves a residual near
# 2.5e-5 rad. Event acceptance uses the final sign-change bracket rather
# than requiring the endpoint value to fall inside this tolerance.
SWITCH_TOL_TIME = 1e-5
SWITCH_TOL_VALUE = 5e-5


def make_fem_parameters():
    sim = config.SimulationParameters()
    sim.tau = 0.001
    sim.t_end = t_end
    sim.with_contact = True
    sim.use_gravity = True

    anim = config.AnimationParameters()
    anim.animate = False
    return {"sim_params": sim, "anim_params": anim}


def build_region_system(label):
    """Closed loop using the default three-model angle-region policy."""
    plant = MasterPendulum(name="MasterPendulum", initial_mode="FMU")
    plant.set_parameters(**{"FEM": make_fem_parameters()})

    plant.initialize(t0=0.0)
    plant.add_event_indicator("wall_hit", func=wall_contact_indicator, direction=-1)
    sp = FMUComponent(name="Setpoint", fmu_path=fmu_paths["Trajectories"]["SetPoint"],
                      group="Reference")
    ctrl = PIDController(name="PID")
    drv = FMUComponent(name="Drive", fmu_path=fmu_paths["Actuators"]["DriveDynamic"],
                       group="Actuator")
    sens = FMUComponent(name="Angle Sensor", fmu_path=fmu_paths["Sensors"]["AngleSensor"],
                        group="Sensors")
    dec = FMUComponent(name="Angle Decoder", fmu_path=fmu_paths["Sensors"]["AngleDecoder"],
                       group="Signal Processing")

    system = System(name=f"Region switching ({label})")
    for comp in (sp, ctrl, drv, plant, dec, sens):
        system.add_component(comp)
    for conn in (
        Connection("Setpoint", "theta_ref", "PID", "theta_ref"),
        Connection("MasterPendulum", "theta", "Angle Sensor", "theta"),
        Connection("Angle Sensor", "v_out", "Angle Decoder", "v_in"),
        Connection("Angle Decoder", "theta", "PID", "theta_meas"),
        Connection("PID", "u", "Drive", "u_control"),
        Connection("Drive", "torque", "MasterPendulum", "tau"),
        Connection("MasterPendulum", "omega", "Drive", "omega"),
    ):
        system.add_connection(conn)
    system.add_event_connection(
        EventConnection("MasterPendulum", "wall_hit", "MasterPendulum", "omega_invert"))
    system.add_event_connection(
        EventConnection("MasterPendulum", "wall_hit", "PID", "resetI"))

    system.initialize(t0=0.0)
    system.algorithm.event_dedup_tol = 5e-4
    system.algorithm.tol_time = SWITCH_TOL_TIME
    system.algorithm.tol_value = SWITCH_TOL_VALUE
    return system, plant

In [ ]:
switch_runs = {}

for variant, macro_dt in (("fine", sim_params.tau), ("coarse", 4 * sim_params.tau)):
    sys_cmp, plant_cmp = build_region_system(variant)
    initial_mode = plant_cmp.active_mode
    sys_cmp.run(0.0, t_end, macro_dt)

    hist = sys_cmp.get_history()
    t_cmp, data_cmp = hist["MasterPendulum"]
    contact_times = [d.t for d in hist["Events"][("MasterPendulum", "wall_hit")]]
    intervals = build_mode_intervals(plant_cmp.sync_events, initial_mode, 0.0, t_end)

    switch_runs[variant] = {
        "t": np.asarray(t_cmp, dtype=float),
        "theta": np.asarray(data_cmp["theta"], dtype=float),
        "switch_times": [float(e["time"]) for e in plant_cmp.sync_events],
        "switch_modes": [f"{e['from_mode']}->{e['to_mode']}" for e in plant_cmp.sync_events],
        "contact_times": contact_times,
        "mode_intervals": intervals,
        "fem_active_s": sum(b - a for a, b, m in intervals if m == "FEM"),
    }

    print(f"{variant:>10}: {len(plant_cmp.sync_events)} switches, "
          f"{len(contact_times)} contact events, "
          f"FEM active {switch_runs[variant]['fem_active_s']:.3f} s")

# The comparison is only valid if both runs resolve the same contact sequence.
n_contacts = {s: len(r["contact_times"]) for s, r in switch_runs.items()}
if len(set(n_contacts.values())) > 1:
    print(f"\nWARNING: contact counts differ {n_contacts}. The trajectories have diverged "
          f"into different bounce sequences, so the error columns below compare different "
          f"physics, not switch placement. Investigate before reading the table.")

In [ ]:
# --- Accuracy against the OpenModelica reference -----------------------
rows = []
for strategy, run in switch_runs.items():
    for interval, bounds in (("full horizon", (None, None)),
                             ("contact window", contact_window)):
        m = trajectory_error_metrics(
            run["t"], run["theta"],
            ref_sol_reset["time"], ref_sol_reset["theta"],
            t_min=bounds[0], t_max=bounds[1],
        )
        rows.append({"switching": strategy, "interval": interval,
                     "E_inf": m["e_inf"], "E_2": m["e_2"], "E_mean": m["e_mean"]})

    ev = event_time_metrics(run["contact_times"], event_times_ref)
    rows.append({"switching": strategy, "interval": "contact event times",
                 "E_inf": ev["max_abs_delta"], "E_2": np.nan, "E_mean": ev["mean_abs_delta"]})

df_switch_cmp = pd.DataFrame(rows)
display(
    df_switch_cmp.style
    .format({"E_inf": "{:.3e}", "E_2": "{:.3e}", "E_mean": "{:.3e}"}, na_rep="—")
    .hide(axis="index")
)

# --- Where the transitions landed --------------------------------------
placement = pd.DataFrame({
    "fine_t": pd.Series(switch_runs["fine"]["switch_times"]),
    "fine_mode": pd.Series(switch_runs["fine"]["switch_modes"]),
    "coarse_t": pd.Series(switch_runs["coarse"]["switch_times"]),
    "coarse_mode": pd.Series(switch_runs["coarse"]["switch_modes"]),
})
placement["shift_s"] = placement["coarse_t"] - placement["fine_t"]

print("\nswitch instants")
print(placement.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print(f"\nFEM active   fine {switch_runs['fine']['fem_active_s']:.3f} s"
      f"   coarse {switch_runs['coarse']['fem_active_s']:.3f} s")
print("Localized switch placement should remain stable as the macro step changes.")

### Trajectory comparison

Panel (a) shows the whole horizon with the mode strips underneath: the upper strip is the
fine-step run, the lower one the coarse-step run. Panel (b) zooms on the contact window, where
the two runs should place their localized transitions consistently. The deviation from the reference is
largest. Panel (c) magnifies a single transition so the placement difference is visible at all —
the localized transitions can be inspected independently of the communication points.

In [ ]:
STRATEGY_STYLE = {
    "fine":   dict(color="#2F4858", linewidth=1.6, linestyle="-",          zorder=4),
    "coarse": dict(color="#B65A3C", linewidth=1.5, linestyle=(0, (4, 2)), zorder=5),
}
MODE_STRIP = {"FMU": "#d6cfc2", "FEM": "#bbd8e8"}

fig = plt.figure(figsize=(FULL_WIDTH, 0.85 * FULL_WIDTH), constrained_layout=True)
gs = fig.add_gridspec(4, 2, height_ratios=[3.0, 0.30, 0.30, 2.2])

ax_full = fig.add_subplot(gs[0, :])
ax_s_grid = fig.add_subplot(gs[1, :], sharex=ax_full)
ax_s_loc = fig.add_subplot(gs[2, :], sharex=ax_full)
ax_contact = fig.add_subplot(gs[3, 0])
ax_zoom = fig.add_subplot(gs[3, 1])

# --- (a) full horizon, (b) contact window ------------------------------
for ax in (ax_full, ax_contact):
    ax.plot(ref_sol_reset["time"], ref_sol_reset["theta"],
            color="0.35", linewidth=1.2, alpha=0.9, label="Reference", zorder=3)
    for strategy, run in switch_runs.items():
        ax.plot(run["t"], run["theta"], label=strategy, **STRATEGY_STYLE[strategy])
    ax.axhline(0, color="gray", linestyle=":", linewidth=1)
    for sign in (+1, -1):
        ax.axhline(sign * SWITCH_THRESHOLD_RAD, color="#B65A3C",
                   linestyle="--", linewidth=0.8, alpha=0.45)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(r"$\theta$ in $\mathrm{rad}$")

ax_full.set_xlim(0, t_end)
ax_full.tick_params(labelbottom=False)
ax_full.legend(loc="upper right", ncol=3, fontsize=8, frameon=True)

# --- mode strips, one per strategy -------------------------------------
for ax, strategy in ((ax_s_grid, "fine"), (ax_s_loc, "coarse")):
    for a, b, mode in switch_runs[strategy]["mode_intervals"]:
        if b <= a:
            continue
        ax.broken_barh([(a, b - a)], (0, 1), facecolors=MODE_STRIP.get(mode, "0.75"),
                       edgecolors="white", linewidth=0.6)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlim(0, t_end)
    ax.set_ylabel(strategy, rotation=0, ha="right", va="center", fontsize=8)
    ax.grid(False)
    for spine in ("left", "right", "top", "bottom"):
        ax.spines[spine].set_visible(False)
ax_s_grid.tick_params(labelbottom=False)
ax_s_loc.tick_params(labelbottom=False)

# --- (b) contact-window zoom -------------------------------------------
ax_contact.set_xlim(*contact_window)
ax_contact.set_ylim(-0.01, 0.09)
ax_contact.set_xlabel(r"$t$ in $\mathrm{s}$")

# --- (c) single-transition zoom ----------------------------------------
# Centre on whichever transition localization moved furthest off the
# communication grid, and span only ~2 macro steps: at a wider span a
# sub-millisecond offset is a few pixels and the panel shows nothing.
def _off_grid(t, tau=sim_params.tau):
    r = t % tau
    return min(r, tau - r)

_loc_switches = switch_runs["fine"]["switch_times"]
_idx = int(np.argmax([_off_grid(t) for t in _loc_switches]))
t_focus = _loc_switches[_idx]
span = 1.2 * sim_params.tau
ax_zoom.plot(ref_sol_reset["time"], ref_sol_reset["theta"],
             color="0.35", linewidth=1.2, alpha=0.9)
for strategy, run in switch_runs.items():
    ax_zoom.plot(run["t"], run["theta"], marker="o", markersize=2.5,
                 **STRATEGY_STYLE[strategy])
    for t_sw in run["switch_times"]:
        if abs(t_sw - t_focus) < span:
            ax_zoom.axvline(t_sw, color=STRATEGY_STYLE[strategy]["color"],
                            linestyle=":", linewidth=1.2)
ax_zoom.axhline(SWITCH_THRESHOLD_RAD, color="#B65A3C", linestyle="--",
                linewidth=0.9, alpha=0.7)
for t_grid in np.arange(t_focus - span, t_focus + span, sim_params.tau):
    ax_zoom.axvline(t_grid, color="0.85", linewidth=0.5, zorder=0)
ax_zoom.set_xlim(t_focus - span, t_focus + span)
ax_zoom.set_xlabel(r"$t$ in $\mathrm{s}$")
ax_zoom.set_ylabel(r"$\theta$ in $\mathrm{rad}$")
ax_zoom.grid(False)

for label, ax in [("(a)", ax_full), ("(b)", ax_contact), ("(c)", ax_zoom)]:
    ax.text(0.02, 0.95, label, transform=ax.transAxes, ha="left", va="top",
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.5))

out = _repo / "notebooks" / "figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "05_c_switching_strategy.pdf")
plt.show()